# 00b — Label Diagnostics

Traces exactly where the OCR → spatial-match → rename chain breaks for a single patent.

Change `patent_id` in **Cell 1** to inspect any patent with many `_Fu` files.

**Pipeline recap (Stage 00)**

```
A  Download pages          →  raw/{id}/fig_01.png, fig_02.png …
B  ocr_all_pages()         →  page_labels dict  (labels + pixel positions)
C  get_brief_description() →  text/{id}.txt
D  segment_patent()        →  fig_02_s01.png, fig_02_s02.png  (originals DELETED)
                              {id}_seg_map.json  (crop bboxes, pre-rename filenames)
E  assign_and_rename_crops()→  {id}_F001.png / {id}_Fu001.png
```

The sidecar JSON stores **pre-rename** filenames.  After step E they no longer exist on disk — this notebook loads the JSON directly rather than re-running `segment_patent()`.

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import sys, json
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.config_loader import load_config

cfg        = load_config()
patent_id  = "US2015014475A1"   # ← change to any patent with many _Fu files

raw_dir    = Path(cfg['paths']['raw_images'])
patent_dir = raw_dir / patent_id

if not patent_dir.exists():
    raise FileNotFoundError(f"Patent folder not found: {patent_dir}")

# ── Inventory ──────────────────────────────────────────────────────────────────
all_files  = sorted(patent_dir.iterdir())
fig_pages  = sorted(patent_dir.glob("fig_[0-9]*.png"))           # original full pages
split_crops= sorted(patent_dir.glob("fig_*_s*.png"))             # pre-rename split crops
fu_files   = sorted(patent_dir.glob(f"{patent_id}_Fu*.png"))     # unlabeled (post-rename)
f_files    = sorted(patent_dir.glob(f"{patent_id}_F[0-9]*.png")) # labeled (post-rename)
seg_json   = patent_dir / f"{patent_id}_seg_map.json"

print(f"Patent     : {patent_id}")
print(f"Folder     : {patent_dir}")
print()
print(f"Original pages (fig_*.png)   : {len(fig_pages)}")
print(f"Pre-rename split crops       : {len(split_crops)}")
print(f"Named crops  (_F*.png)       : {len(f_files)}")
print(f"Unlabeled    (_Fu*.png)      : {len(fu_files)}")
print(f"Seg-map JSON                 : {'EXISTS' if seg_json.exists() else 'MISSING'}")
print()
print("All files in folder:")
for f in all_files:
    print(f"  {f.name}")

In [ ]:
# ── Cell 2: Show full pages (or warn if deleted) ───────────────────────────────
import matplotlib.pyplot as plt
from PIL import Image

if not fig_pages:
    print("⚠️  ORIGINAL PAGES DELETED — this is likely the root cause.")
    print("   segment_patent() deletes fig_*.png after extracting sub-figure crops.")
    print("   Without the original pages we cannot re-run page-level OCR (Cell 3).")
    print()
    print("   To get original pages again: delete the patent folder and re-run Stage 00.")
    print()

    # We can still show the seg_map structure to understand what was there
    if seg_json.exists():
        seg_raw = json.loads(seg_json.read_text())
        print("Seg-map structure (pages + crop counts):")
        for page_stem, crops_data in seg_raw.items():
            bboxed = sum(1 for v in crops_data.values() if v is not None)
            print(f"  {page_stem}: {len(crops_data)} crop(s), {bboxed} with bbox")
    else:
        print("   No seg_map.json — patent may have been processed without the bbox patch.")
else:
    print(f"Found {len(fig_pages)} original page(s):")
    for page in fig_pages:
        print(f"  {page.name}")
    print()

    n_cols = min(len(fig_pages), 3)
    n_rows = (len(fig_pages) + n_cols - 1) // n_cols
    fig_plot, axes = plt.subplots(n_rows, n_cols,
                                  figsize=(6 * n_cols, 8 * n_rows),
                                  squeeze=False)
    for idx, page in enumerate(fig_pages):
        ax = axes[idx // n_cols][idx % n_cols]
        img = Image.open(page)
        ax.imshow(img, cmap='gray' if img.mode == 'L' else None)
        ax.set_title(page.name, fontsize=9)
        ax.axis('off')
    for idx in range(len(fig_pages), n_rows * n_cols):
        axes[idx // n_cols][idx % n_cols].axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Cell 3: Re-run page-level OCR ─────────────────────────────────────────────
#
# ocr_all_pages() must run on the ORIGINAL pages (before HR-Net).  If they are
# gone this cell shows what the empty result looks like so Cell 5 still runs.

from src.ocr_labeler import ocr_all_pages

page_labels: dict = {}

if not fig_pages:
    print("⚠️  No original pages — skipping OCR.")
    print("   page_labels will be empty; Cell 5 spatial simulation cannot run.")
else:
    print(f"Running page-level OCR on {len(fig_pages)} page(s)...")
    page_labels = ocr_all_pages(fig_pages, cfg)
    print()

    any_found = any(v for v in page_labels.values())
    if not any_found:
        print("⚠️  OCR found NO 'FIG. N' labels on ANY page.")
        print("   Likely cause: cursive/stylized font that Tesseract cannot read.")
        print("   Displaying pages for visual inspection:")
        for page in fig_pages:
            img = Image.open(page)
            fig_plot, ax = plt.subplots(figsize=(7, 9))
            ax.imshow(img, cmap='gray' if img.mode == 'L' else None)
            ax.set_title(f"OCR returned nothing: {page.name}", color='red')
            ax.axis('off')
            plt.tight_layout()
            plt.show()
    else:
        for page_stem, labels in page_labels.items():
            if labels:
                print(f"  {page_stem}: {len(labels)} label(s) found")
                for label, cx, cy in labels:
                    print(f"    '{label}'  at pixel ({cx:>5}, {cy:>5})")
            else:
                print(f"  {page_stem}: (no labels)")

In [ ]:
# ── Cell 4: Show _Fu crops with bbox data from seg_map ────────────────────────
#
# The seg_map stores PRE-RENAME filenames (fig_02_s01.png) paired with their
# bbox in original page coordinates.  After assign_and_rename_crops() these
# become _Fu001.png etc., so direct name lookup fails.  We show the map
# structure and the crops side-by-side.

import matplotlib.patches as mpatches

# ── Load seg_map ────────────────────────────────────────────────────────────────
seg_map_raw: dict = {}          # page_stem → {pre_rename_name: bbox_or_none}
pre_rename_bboxes: dict = {}    # pre_rename_name → bbox

if seg_json.exists():
    seg_map_raw = json.loads(seg_json.read_text())
    for page_stem, crops_data in seg_map_raw.items():
        for fname, bbox_list in crops_data.items():
            pre_rename_bboxes[fname] = tuple(bbox_list) if bbox_list else None
    print(f"Seg-map loaded: {len(pre_rename_bboxes)} crop entries")
    print()
    for page_stem, crops_data in seg_map_raw.items():
        print(f"  {page_stem}:")
        for fname, bbox in crops_data.items():
            bbox_str = str(bbox) if bbox else "None (single-figure page)"
            print(f"    {fname:<35}  bbox: {bbox_str}")
else:
    print("⚠️  No seg_map.json — bbox data unavailable.")
    print("   This patent was likely processed before the bbox-sidecar feature was added.")
    print("   Re-run Stage 00 from scratch to generate spatial-matching data.")

print()
print(f"Displaying up to 30 _Fu crops  ({len(fu_files)} total):")
print()

MAX_DISPLAY = 30
n_show = min(len(fu_files), MAX_DISPLAY)
if n_show == 0:
    print("  No _Fu files found.")
else:
    cols = 5
    rows = (n_show + cols - 1) // cols
    fig_plot, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows), squeeze=False)
    for idx, fu_path in enumerate(fu_files[:n_show]):
        ax  = axes[idx // cols][idx % cols]
        img = Image.open(fu_path)
        ax.imshow(img, cmap='gray' if img.mode == 'L' else None)
        ax.set_title(fu_path.name, fontsize=6, wrap=True)
        ax.axis('off')
    for idx in range(n_show, rows * cols):
        axes[idx // cols][idx % cols].axis('off')
    plt.suptitle(f"_Fu crops for {patent_id}", fontsize=11)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Cell 5: Spatial match simulation ──────────────────────────────────────────
#
# Manually calls _spatial_match_labels() with the actual OCR labels and bboxes.
# This shows exactly which crop got which label and why assignments returned None.
#
# Requirements: page_labels must be non-empty (Cell 3 ran on original pages)
#               seg_map_raw must exist (Cell 4 loaded the sidecar JSON)

from src.ocr_labeler import _spatial_match_labels

any_page_labels = any(v for v in page_labels.values())

if not any_page_labels and not page_labels:
    print("⚠️  page_labels is empty because original pages were deleted.")
    print("   Cannot simulate spatial matching without OCR position data.")
    print("   Re-download the patent and re-run Stage 00 to test the fix.")
elif not any_page_labels:
    print("⚠️  Page-level OCR found no labels — spatial matching never had data to work with.")
    print("   Root cause: OCR failure (cursive font or low contrast).")
elif not seg_map_raw:
    print("⚠️  No seg_map bboxes available — cannot simulate spatial matching.")
    print("   Re-run Stage 00 from scratch to generate bbox data.")
else:
    print("Simulating spatial label assignment per page:\n")

    for page_stem, crops_data in seg_map_raw.items():
        labels = page_labels.get(page_stem, [])

        # Reconstruct (path, bbox) list in seg_map order
        crops_with_bboxes = [
            (patent_dir / fname,
             tuple(bbox) if bbox else None)
            for fname, bbox in crops_data.items()
        ]

        n_crops = len(crops_with_bboxes)
        print(f"  Page '{page_stem}' — {n_crops} crop(s), {len(labels)} OCR label(s)")

        if not labels:
            for fname in crops_data:
                print(f"    {fname:<40} → None  (no OCR labels on this page)")
            print()
            continue

        if n_crops <= 1:
            # Single-figure page: sequential consumption (no spatial matching needed)
            fname = list(crops_data.keys())[0]
            print(f"    {fname:<40} → {labels[0][0]}  (sequential, single-figure page)")
            print()
            continue

        # Multi-crop page: simulate spatial matching
        assignments = _spatial_match_labels(crops_with_bboxes, labels)

        for (crop_path, bbox), assigned in zip(crops_with_bboxes, assignments):
            bbox_str = f"({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]})" if bbox else "None"
            result   = assigned if assigned else "→ None  ← MISS"
            marker   = "  " if assigned else "⚠ "
            print(f"  {marker}{crop_path.name:<40} bbox={bbox_str}  →  {result}")

            if not assigned and labels:
                # Explain why: show distances to all labels
                if bbox:
                    x1, y1, x2, y2 = bbox
                    crop_h = max(y2 - y1, 1)
                    for lbl, lx, ly in labels:
                        # Replicate _spatial_match_labels distance logic
                        above_limit = ly < y1 - crop_h
                        dx = max(x1 - lx, 0, lx - x2)
                        dy = max(y1 - ly, 0, ly - y2)
                        dist = (dx*dx + dy*dy)**0.5
                        why = " (SKIPPED: too far above)" if above_limit else f" dist={dist:.0f}px"
                        print(f"       label '{lbl}' at ({lx},{ly}){why}")
                else:
                    print(f"       crop has no bbox — spatial match cannot run")
        print()

In [ ]:
# ── Cell 6: Summary diagnosis ──────────────────────────────────────────────────
#
# Classifies each _Fu crop into one of five failure reasons:
#   original_page_deleted  — full page no longer on disk
#   cursive_font_ocr_fail  — page exists but OCR returned nothing
#   label_outside_bbox     — OCR found label but spatial match returned None
#   bbox_missing           — crop has no bbox (single-figure page path)
#   unknown                — could not determine

import pandas as pd

# Build a flat list of pre-rename crops with bboxes (ordered by page, then crop index)
# We can't reliably map pre-rename → post-rename names, but we can categorise by
# page and determine per-page failure reason — which is the actionable insight.

any_page_labels = any(v for v in page_labels.values())

rows = []

if not seg_map_raw:
    # No seg_map at all — classify all _Fu as unknown or page_deleted
    reason = "original_page_deleted" if not fig_pages else "unknown"
    for fu in fu_files:
        rows.append({
            "crop_file":            fu.name,
            "bbox_available":       False,
            "page_ocr_found_label": any_page_labels,
            "spatial_match_result": "N/A",
            "failure_reason":       reason,
        })
else:
    # Walk seg_map in document order and classify each pre-rename crop
    # Note: post-rename _Fu names cannot be mapped back, so we report
    # pre-rename names in the table (more informative for diagnosis).
    for page_stem, crops_data in seg_map_raw.items():
        labels       = page_labels.get(page_stem, [])
        page_ocr_ok  = bool(labels)
        n_crops      = len(crops_data)

        if n_crops > 1 and page_ocr_ok:
            # Run spatial match to get per-crop result
            cwb = [
                (patent_dir / fname, tuple(bbox) if bbox else None)
                for fname, bbox in crops_data.items()
            ]
            assignments = _spatial_match_labels(cwb, labels)
        else:
            assignments = [None] * n_crops

        for (fname, bbox_list), assigned in zip(crops_data.items(), assignments):
            bbox_available = bbox_list is not None

            if not fig_pages and not page_labels:
                failure = "original_page_deleted"
            elif not page_ocr_ok:
                if fig_pages:
                    failure = "cursive_font_ocr_fail"
                else:
                    failure = "original_page_deleted"
            elif not bbox_available and n_crops == 1:
                # Single-figure page should have been sequential-labeled
                # If it ended up _Fu anyway → OCR on the crop itself failed
                failure = "cursive_font_ocr_fail"
            elif not bbox_available:
                failure = "bbox_missing"
            elif assigned is not None:
                # Spatial match DID assign — but the crop became _Fu anyway.
                # This can happen if crop-level OCR also ran and overrode it.
                failure = "unknown"
            else:
                failure = "label_outside_bbox"

            rows.append({
                "crop_file":            fname,
                "bbox_available":       bbox_available,
                "page_ocr_found_label": page_ocr_ok,
                "spatial_match_result": assigned if assigned else "None",
                "failure_reason":       failure,
            })

diag_df = pd.DataFrame(rows) if rows else pd.DataFrame(
    columns=["crop_file", "bbox_available", "page_ocr_found_label",
             "spatial_match_result", "failure_reason"]
)

print("=" * 72)
print(f"Diagnostic summary for: {patent_id}")
print("=" * 72)
print(f"  Total _Fu crops         : {len(fu_files)}")
print(f"  Seg-map entries analysed: {len(diag_df)}")
print()
print("Failure reason breakdown:")
if len(diag_df):
    for reason, count in diag_df['failure_reason'].value_counts().items():
        pct = 100 * count / len(diag_df)
        print(f"  {reason:<30}  {count:>4}  ({pct:.0f}%)")
print()

REASON_FIXES = {
    "original_page_deleted":  "Re-download and re-run Stage 00 from scratch.",
    "cursive_font_ocr_fail":  "Strategy 4 (Otsu+PSM7/8) in ocr_figure_label() addresses this.",
    "label_outside_bbox":     "_spatial_match_labels() should catch this — check crop_h threshold.",
    "bbox_missing":           "Legacy run — no sidecar JSON.  Re-run to get bbox data.",
    "unknown":                "Manual inspection needed.",
}
if len(diag_df):
    top_reason = diag_df['failure_reason'].value_counts().index[0]
    print(f"Primary root cause : {top_reason}")
    print(f"Recommended fix    : {REASON_FIXES.get(top_reason, 'See table.')}")
    print()

print("Full diagnosis table:")
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.width', 120)
print(diag_df.to_string(index=False))